# Imports

In [1]:
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/X_train_stacking_layer_one.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test = pd.read_parquet('../data/X_test_stacking_layer_one.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,extra_0,extra_1,extra_2,rf_0,rf_1,rf_2
0,0.999956,0.000041,2.488056e-06,0.999942,0.000036,2.164097e-05,0.999928,0.000068,3.860194e-06,0.999953,0.000044,3.478719e-06,0.930569,0.007605,0.061826,0.999910,0.000056,0.000034
1,0.988684,0.000261,1.105495e-02,0.982394,0.000034,1.757179e-02,0.992420,0.000523,7.057480e-03,0.983714,0.000841,1.544465e-02,0.819961,0.005144,0.174894,0.970685,0.000350,0.028965
2,0.000002,0.999998,9.047677e-08,0.000022,0.999978,1.357429e-07,0.000002,0.999997,1.084726e-07,0.000001,0.999999,9.847311e-08,0.007812,0.961837,0.030351,0.000011,0.999979,0.000011
3,0.999884,0.000113,3.403338e-06,0.999815,0.000182,2.531501e-06,0.999781,0.000213,6.182258e-06,0.999371,0.000627,2.417958e-06,0.943643,0.006641,0.049717,0.999213,0.000571,0.000216
4,0.999361,0.000626,1.369612e-05,0.998781,0.001195,2.377896e-05,0.999596,0.000392,1.163140e-05,0.992451,0.007527,2.222359e-05,0.899693,0.009009,0.091297,0.998640,0.000989,0.000371


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,extra_0,extra_1,extra_2,rf_0,rf_1,rf_2
0,0.999184,0.000763,0.000053,0.997587,0.001932,0.000480,0.998360,0.001543,0.000097,0.998873,0.001052,0.000075,0.665138,0.066561,0.268301,0.987747,0.006744,0.005509
1,0.997347,0.002648,0.000006,0.998597,0.001402,0.000001,0.998518,0.001476,0.000006,0.991409,0.008583,0.000008,0.948686,0.016400,0.034914,0.997904,0.001933,0.000163
2,0.998253,0.000180,0.001567,0.999326,0.000013,0.000661,0.998020,0.000144,0.001837,0.992865,0.000330,0.006805,0.604564,0.021978,0.373458,0.977949,0.003788,0.018263
3,0.001209,0.000315,0.998475,0.001704,0.000080,0.998216,0.000952,0.000309,0.998739,0.000192,0.000183,0.999626,0.072561,0.055581,0.871859,0.008787,0.004687,0.986526
4,0.999902,0.000093,0.000006,0.999799,0.000198,0.000002,0.999733,0.000259,0.000008,0.999429,0.000553,0.000019,0.956355,0.012354,0.031291,0.999828,0.000095,0.000077


# Machine Learning

In [7]:
models = dict(
    lgbm=load_pickle('../models/layer_2/model_lightgbm.pkl'),
    cat=load_pickle('../models/layer_2/model_catboost.pkl'),
    xgb=load_pickle('../models/layer_2/model_xgboost.pkl'),
    hist=load_pickle('../models/layer_2/model_hist_gradient_boosting.pkl'),
    rf=load_pickle('../models/layer_2/model_random_forest.pkl'),
    extra=load_pickle('../models/layer_2/model_extra_tree.pkl'),
)

## Train Dataset

In [8]:
cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=5)

In [9]:
X_train_stacking = pd.DataFrame({})

In [10]:
for model_name, model in tqdm(models.items()):
    
    print(f"Predicting Train Dataset {model_name}")

    predictions = cross_val_predict(model, X_train, y_train.class_encoded, cv=cv, n_jobs=-1, method='predict_proba')
    X_train_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = predictions

  0%|                                                                                                                                                                                         | 0/6 [00:00<?, ?it/s]

Predicting Train Dataset lgbm


 17%|█████████████████████████████▎                                                                                                                                                  | 1/6 [03:32<17:44, 212.86s/it]

Predicting Train Dataset cat


 33%|██████████████████████████████████████████████████████████▋                                                                                                                     | 2/6 [11:29<24:31, 367.93s/it]

...

 67%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▎                                                          | 4/6 [13:15<05:03, 151.86s/it]

Predicting Train Dataset rf


 83%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▋                             | 5/6 [27:56<06:54, 414.56s/it]

Predicting Train Dataset extra


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [29:04<00:00, 290.68s/it]


## Test Dataset

In [15]:
X_test_stacking = pd.DataFrame({})

In [16]:
for model_name, model in tqdm(models.items()):
    
    print(f"Predicting Test Dataset {model_name}")
    
    X_test_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = model.predict_proba(X_test)

  0%|                                                                                                                                                                                         | 0/6 [00:00<?, ?it/s]

Predicting Test Dataset lgbm


 17%|█████████████████████████████▌                                                                                                                                                   | 1/6 [00:32<02:43, 32.60s/it]

Predicting Test Dataset cat


 33%|███████████████████████████████████████████████████████████                                                                                                                      | 2/6 [00:32<00:54, 13.57s/it]

...

 67%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████                                                           | 4/6 [00:33<00:09,  4.75s/it]

Predicting Test Dataset rf


 83%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌                             | 5/6 [00:35<00:03,  3.61s/it]

Predicting Test Dataset extra


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:36<00:00,  6.13s/it]


# Saving

In [17]:
X_train_stacking.to_parquet('../data/X_train_stacking_layer_two.parquet')
X_test_stacking.to_parquet('../data/X_test_stacking_layer_two.parquet')

In [18]:
X_train_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.999980,0.000019,7.173664e-07,0.999940,0.000059,0.000001,0.999902,0.000087,0.000011,0.999964,0.000035,9.391767e-07,9.999364e-01,0.000064,0.000000,0.998533,0.001276,0.000191
1,0.993270,0.000283,6.446509e-03,0.994368,0.000245,0.005387,0.993982,0.000245,0.005772,0.992230,0.000265,7.504966e-03,9.951892e-01,0.000171,0.004639,0.972023,0.002391,0.025586
2,0.000034,0.999962,4.381048e-06,0.000139,0.999859,0.000002,0.000214,0.999774,0.000012,0.000013,0.999981,6.276192e-06,3.944814e-07,1.000000,0.000000,0.000114,0.999862,0.000024
3,0.999879,0.000120,7.617780e-07,0.999812,0.000187,0.000001,0.999843,0.000145,0.000012,0.999931,0.000069,5.789762e-07,9.998199e-01,0.000180,0.000000,0.998349,0.001455,0.000196
4,0.997913,0.002069,1.828613e-05,0.998790,0.001202,0.000008,0.998263,0.001704,0.000033,0.996330,0.003666,3.709336e-06,9.977873e-01,0.002205,0.000008,0.994559,0.004657,0.000784


In [19]:
X_test_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,cat_0,cat_1,cat_2,xgb_0,xgb_1,xgb_2,hist_0,hist_1,hist_2,rf_0,rf_1,rf_2,extra_0,extra_1,extra_2
0,0.998106,0.001851,0.000042,0.997287,0.002522,0.000192,0.998122,0.001845,0.000033,0.993568,0.006380,0.000053,0.998112,0.001858,0.000030,0.991100,0.006021,0.002879
1,0.997635,0.002352,0.000014,0.997773,0.002224,0.000003,0.997897,0.002071,0.000032,0.996850,0.003147,0.000003,0.998374,0.001623,0.000003,0.993814,0.005762,0.000424
2,0.997136,0.001323,0.001541,0.997897,0.000212,0.001892,0.997850,0.000528,0.001622,0.992689,0.000960,0.006351,0.997662,0.000721,0.001618,0.987143,0.003596,0.009262
3,0.000485,0.000066,0.999449,0.001142,0.000192,0.998666,0.000653,0.000081,0.999266,0.000062,0.000010,0.999928,0.000413,0.000015,0.999572,0.000552,0.000322,0.999126
4,0.999485,0.000510,0.000004,0.999636,0.000360,0.000003,0.999666,0.000319,0.000016,0.999455,0.000531,0.000014,0.999776,0.000222,0.000002,0.998133,0.001603,0.000264
